This is the notebook to run inference on trained model using the `CoppeliaSim` robot simulator. Follow the below steps for running this notebook

1. Download and install CoppeliaSim from here: [https://www.coppeliarobotics.com/](https://www.coppeliarobotics.com/)
2. Open the scene file `\simulation\uav\coppeliasim_envs\uav_common_env.ttt` in the simulator software (must do it before running the notebook).
3. Run all the cells in the notebook (there is an assertion error just to prevent running the notebook before opening the scene file when using the run all option. Please continue running below cells if you get assertion error.)
4. For every environment, modify the path for trained model and run the cell.
4. **IMPORTANT**: When closing the CoppeliaSim scene file, please select "No" for the question "Do you wish to save the changes?"

### Load modules and configurations

In [ ]:
import os
import copy
import json
import numpy as np
import gymnasium as gym
import pygame

from sb3_contrib import CrossQ
from coppeliasim_zmqremoteapi_client import RemoteAPIClient

In [ ]:
# ================================
# Utility / Helper Functions
# ================================
def is_inside_polygon(point, poly):  # check if a point is inside polygon
    x, y = point  # unpack point
    inside = False  # initialize flag
    n = len(poly)  # number of vertices
    p1x, p1y = poly[0]  # first vertex
    for i in range(n + 1):  # iterate through edges
        p2x, p2y = poly[i % n]  # next vertex (wrap around)
        # check if point is within vertical bounds of edge
        if min(p1y, p2y) < y <= max(p1y, p2y) and x <= max(p1x, p2x):
            if p1y != p2y:  # avoid division by zero
                xinters = (y - p1y) * (p2x - p1x) / (p2y - p1y) + p1x  # intersection
            if p1x == p2x or x <= xinters:  # crossing check
                inside = not inside  # toggle
        p1x, p1y = p2x, p2y  # move to next edge
    return inside  # return result

def compute_min_dist(x):  # compute minimum pairwise distance
    x = np.asarray(x, dtype=np.float32)  # ensure array
    diff = x[:, None, :] - x[None, :, :]  # pairwise differences
    dist_matrix = np.linalg.norm(diff, axis=-1)  # compute distances
    np.fill_diagonal(dist_matrix, np.inf)  # ignore self-distance
    return float(np.min(dist_matrix))  # return min distance

def binary_list_to_decimal(bin_list):
    """Convert a list of 0/1 values to its decimal integer equivalent."""
    return int("".join(str(int(b)) for b in bin_list), 2)

def read_uav_json(json_path, sf=1):
    """Load UAV field-info dicts from JSON, applying a scale factor sf."""
    with open(json_path, "r") as f:
        data = json.load(f)
    for _, cfg in data.items():
        cfg["field"]             = [tuple((p[0] * sf, p[1] * sf)) for p in cfg["field"]]
        cfg["init_positions"]    = [np.array(p, dtype=float) * sf for p in cfg["init_positions"]]
        cfg["infected_locations"]= [tuple((p[0] * sf, p[1] * sf)) for p in cfg["infected_locations"]]
    return data


Make the multi-agent class:

In [ ]:
class MultiUAV(gym.Env):
    """
    Multi-UAV path-planning environment.
    Robots visit every infected/target location (binary coverage, no spraying).
    """
    metadata = {'render_modes': ['human', 'rgb_array'], "render_fps": 60}
    
    def __init__(
        self,
        field_info,
        render_mode=None,
        wind_par=None,
        num_robots=3,
        max_steps=1000,
        target_screen_size=800,
        # ── experiment control ──────────────────────────────────────
        reward_ablation="full",
        obs_mode="full",
        uncertainty_mode="full",
        dr_mode="none"
    ):
        super().__init__()

        # ── Validate experiment parameters ──────────────────────────
        assert uncertainty_mode in ("full", "wind_only", "act_only", "deterministic"), \
            f"Unknown uncertainty_mode: {uncertainty_mode}"
        assert dr_mode in ("none", "wind", "full"), \
            f"Unknown dr_mode: {dr_mode}"
        assert reward_ablation in ("full", "no_term", "no_path"), \
            f"Unknown reward_ablation: {reward_ablation}"
        assert obs_mode in ("full", "no_pos", "no_vis_hist", "pos_only"), \
            f"Unknown obs_mode: {obs_mode}"
        assert render_mode is None or render_mode in self.metadata["render_modes"]
        
        # Store experiment flags
        self.reward_ablation  = reward_ablation
        self.obs_mode         = obs_mode
        self.uncertainty_mode = uncertainty_mode
        self.dr_mode          = dr_mode
        self.max_steps        = max_steps

        # ── Field info ──────────────────────────────────────────────
        self.field_info    = copy.deepcopy(field_info)          # Deep copy to prevent mutating original field data
        self.poly_vertices = self.field_info['field']           # Extract polygon vertices defining the boundary
        xs, ys             = zip(*self.poly_vertices)           # Unzip coordinates into separate X and Y tuples
        self.min_x, self.max_x = float(np.min(xs)), float(np.max(xs)) # Find min/max X for bounding box
        self.min_y, self.max_y = float(np.min(ys)), float(np.max(ys)) # Find min/max Y for bounding box
        self.world_width   = self.max_x - self.min_x            # Calculate total width of the environment
        self.world_height  = self.max_y - self.min_y            # Calculate total height of the environment

        # ── Rendering setup ─────────────────────────────────────────
        # Auto-scale render window to fit the target screen size
        scale_x = target_screen_size / self.world_width
        scale_y = target_screen_size / self.world_height
        self.render_scale  = min(scale_x, scale_y) * 0.90       # Scale down slightly (90%) to leave margins
        self.screen_width  = int(self.world_width  * self.render_scale) + 40  # Screen width with padding
        self.screen_height = int(self.world_height * self.render_scale) + 40  # Screen height with padding
        self.offset_x      = (self.screen_width  - self.world_width  * self.render_scale) / 2 # Center X offset
        self.offset_y      = (self.screen_height - self.world_height * self.render_scale) / 2 # Center Y offset

        # ── Robot params ────────────────────────────────────────────
        self.num_robots   = num_robots                          # Total number of UAVs
        self.init_robot_positions = np.array(
            self.field_info['init_positions'][:num_robots], dtype=np.float32) # Fetch initial positions
        self.robot_size   = 1.0                                 # Base robot visual/collision size
        self.mass         = 1.0                                 # UAV mass (overridden by DR full)
        self.thrust_power = 0.5                                 # Action scaling multiplier (overridden by DR full)
        self.max_speed    =  5.0                                # Maximum allowable speed
        self.min_speed    = -5.0                                # Minimum allowable speed
        self.robot_colors = [                                   # Color palette for differentiating robots
            (255, 0,   0), (0, 200,   0), (0,   0, 255),
            (255, 128, 0), (128, 0, 255), (255, 0, 255), (128, 128, 128),
        ]

        # ── Infection / Target params ───────────────────────────────
        self.initial_inf_locations  = [tuple(loc) for loc in self.field_info['infected_locations']] # Target coordinates
        self._nominal_infected_size = 1.5                       # Base radius for successful visitation
        self.infected_size          = self._nominal_infected_size
        self.infected_length        = len(self.initial_inf_locations) # Total number of targets

        # ── Base wind (mean) magnitude and direction ────────────────
        if wind_par is None: # Load wind parameters if not provided
            wind_par = [0, 0]
        self.base_wind_mag, self.base_wind_dir = float(wind_par[0]), float(wind_par[1])

        # ── Noise stds — set by uncertainty_mode ────────────────────
        # These are the *nominal* values; DR "full" may override at the start of each episode.
        _noise = {
            "full":          dict(wind=0.20, wind_dir=5.0, action=0.10, obs=0.01),
            "wind_only":     dict(wind=0.20, wind_dir=5.0, action=0.00, obs=0.00),
            "act_only":      dict(wind=0.00, wind_dir=0.0, action=0.10, obs=0.00),
            "deterministic": dict(wind=0.00, wind_dir=0.0, action=0.00, obs=0.00),
        }[uncertainty_mode]

        self.wind_noise_std      = _noise["wind"]               # Wind magnitude volatility
        self.wind_dir_noise_std  = _noise["wind_dir"]           # Wind direction volatility
        self.action_noise_std    = _noise["action"]             # Volatility applied to UAV control inputs
        self.obs_noise_std       = _noise["obs"]                # Sensor noise added to state observations
        self.init_position_noise = 0.05                          # Jitter added to spawn coordinates

        # ── Nominal values for DR restore ───────────────────────────
        self._nominal_action_noise_std = _noise["action"]
        self._nominal_mass             = 1.0
        self._nominal_thrust_power     = 0.5

        # ── Action space: (ax, ay) per robot ────────────────────────
        # Each robot's action has the following:
        #   1. a_x = Force component (or thrust) along x-axis
        #   2. a_y = Force component (or thrust) along y-axis
        # Therefore, total actions = 2 * num_robots
        self.action_space = gym.spaces.Box(
            low=-1.0, high=1.0, shape=(num_robots, 2), dtype=np.float32)

        # ── Observation space — depends on obs_mode ─────────────────
        # "full"        positions(2N) + velocities(2N) + visited_decimal(1) = 4N+1
        # "no_pos"      visited_decimal(1)  [remove all kinematics]
        # "no_vis_hist" positions(2N) + velocities(2N) = 4N
        # "pos_only"    positions(2N)
        N = num_robots
        _obs_dims = {"full": 4*N+1, "no_pos": 1, "no_vis_hist": 4*N, "pos_only": 2*N}
        self.observation_space = gym.spaces.Box(
            low=-np.inf, high=np.inf, shape=(_obs_dims[obs_mode],), dtype=np.float32)

        self.render_mode = render_mode
        self.screen      = None
        self.clock       = None
        self.reset() # Initialise state

    # ── coordinate conversion ────────────────────────────────────────
    def world_to_screen(self, pos):
        """Converts physical world coordinates to PyGame screen coordinates."""
        x = (pos[0] - self.min_x) * self.render_scale + self.offset_x
        y = (pos[1] - self.min_y) * self.render_scale + self.offset_y
        return int(x), int(y)

    # ── observation builder ──────────────────────────────────────────
    def _get_obs(self):
        """Constructs the state array based on the selected observation mode."""
        # Convert the binary list of visited locations into a single float for the neural network
        infected_decimal = float(binary_list_to_decimal(list(self.infected_dict.values())))
        
        if self.obs_mode == "full":
            state = np.concatenate([self.robot_positions.flatten(),
                                    self.robot_velocities.flatten(),
                                    np.array([infected_decimal], dtype=np.float32)])
        elif self.obs_mode == "no_pos":
            state = np.array([infected_decimal], dtype=np.float32)
        elif self.obs_mode == "no_vis_hist":
            state = np.concatenate([self.robot_positions.flatten(),
                                    self.robot_velocities.flatten()])
        elif self.obs_mode == "pos_only":
            state = self.robot_positions.flatten().copy()

        state = state.astype(np.float32)
        
        # Add observation noise if specified
        if self.obs_noise_std > 0:
            state += np.random.normal(0, self.obs_noise_std, size=state.shape).astype(np.float32)

        # Package cleanly formatted info dictionary
        info = {f'robot{i}': self.robot_positions[i].copy() for i in range(self.num_robots)}
        return state, info

    # ── reset ────────────────────────────────────────────────────────
    def reset(self, seed=None, options=None):
        """Resets the environment for a new episode."""
        super().reset(seed=seed)

        # ── Domain randomization — re-sample base params each episode ─
        if self.dr_mode == "none":
            # Standard: add small episode-level noise to the nominal wind
            self.wind_mag         = self.base_wind_mag + np.random.normal(0, self.wind_noise_std)
            self.wind_dir         = self.base_wind_dir + np.random.normal(0, self.wind_dir_noise_std)
            # Restore nominal physical params (may have been overridden last episode)
            self.action_noise_std = self._nominal_action_noise_std
            self.infected_size    = self._nominal_infected_size
            self.mass             = self._nominal_mass
            self.thrust_power     = self._nominal_thrust_power
            
        elif self.dr_mode == "wind":
            # Randomise wind speed and direction uniformly
            self.wind_mag         = float(np.random.uniform(0.0, 1.0))
            self.wind_dir         = float(np.degrees(np.random.uniform(0.0, 2 * np.pi)))
            self.action_noise_std = self._nominal_action_noise_std
            self.infected_size    = self._nominal_infected_size
            self.mass             = self._nominal_mass
            self.thrust_power     = self._nominal_thrust_power
            
        elif self.dr_mode == "full":
            # Randomise all physical parameters significantly to improve policy robustness
            self.wind_mag         = float(np.random.uniform(0.0, 1.0))
            self.wind_dir         = float(np.degrees(np.random.uniform(0.0, 2 * np.pi)))
            self.action_noise_std = float(np.random.uniform(0.01, 0.10))
            r0                    = self._nominal_infected_size
            self.infected_size    = float(np.random.uniform(0.8 * r0, 1.2 * r0))
            self.mass             = float(np.random.uniform(0.90, 1.10))
            self.thrust_power     = 0.5 * float(np.random.uniform(0.80, 1.20))

        # ── Randomise starting positions ─────────────────────────────
        self.robot_positions = (
            self.init_robot_positions
            + np.random.normal(0, self.init_position_noise, self.init_robot_positions.shape)
        ).astype(np.float32)

        # ── Initialise dynamic state ─────────────────────────────────
        self.step_count         = 0
        self.visited            = set()                                 # Cells currently visited to track exploration
        self.infected_locations = list(copy.deepcopy(self.initial_inf_locations)) # Remaining targets
        self.infected_dict      = {loc: 0 for loc in self.initial_inf_locations}  # 0=unvisited, 1=visited
        self.robot_velocities   = np.zeros((self.num_robots, 2), dtype=np.float32)
        
        # ── Episode counters ─────────────────────────────────────────
        self.trajectories       = [[] for _ in range(self.num_robots)]  # Breadcrumbs for rendering
        self.total_path_length  = 0.0                                   # Accumulator for distance travelled
        self.prev_positions     = self.robot_positions.copy()           # Memory of previous step for path diff

        return self._get_obs()

    # ── step ─────────────────────────────────────────────────────────
    def step(self, actions):
        """Advances the simulation by one timestep."""
        self.step_count += 1
        terminated, truncated = False, False
        rewards = 0.0

        # ── Stochastic wind for this step ────────────────────────────
        # Wind changes dynamically every frame based on noise standard deviation
        wind_mag = self.wind_mag + np.random.normal(0, self.wind_noise_std)
        wind_dir = self.wind_dir + np.random.normal(0, self.wind_dir_noise_std)
        theta_w  = np.radians(wind_dir)
        wind     = np.array([wind_mag * np.cos(theta_w), wind_mag * np.sin(theta_w)],
                            dtype=np.float32)

        for i in range(self.num_robots):                        # For each robot
            ax_raw, ay_raw = actions[i]                         # Get raw neural net outputs

            # Action noise + scaling
            ax = ax_raw * self.thrust_power + np.random.normal(0, self.action_noise_std)
            ay = ay_raw * self.thrust_power + np.random.normal(0, self.action_noise_std)

            # Velocity update (Newtonian dynamics F=ma -> a=F/m)
            self.robot_velocities[i] += np.array([ax, ay]) / self.mass + wind
            self.robot_velocities[i]  = np.clip(self.robot_velocities[i], self.min_speed, self.max_speed)

            # Position update
            new_pos = self.robot_positions[i] + self.robot_velocities[i] # Predict next position
            if is_inside_polygon(new_pos, self.poly_vertices):           # Check if new position is inside the field
                self.robot_positions[i] = new_pos                        # Move to the new location
            else:
                rewards -= 50                                            # Boundary penalty — unchanged vs reference env
                self.robot_velocities[i][:] = 0                          # Set robot velocities to zero (crash into wall)

            # ── Per-robot movement penalties ─────────────────
            if self.reward_ablation != "no_path":
                rewards -= 0.1 * (ax ** 2 + ay ** 2)                     # Energy penalty (encourages efficient control)
                rewards -= 0.3 * float(np.linalg.norm(self.robot_velocities[i]))  # Speed penalty (Penalize high speeds)

            # ── Coverage shaping ─────────────────────────────────────────────────────
            # Encourage exploring new areas by mapping continuous space to a grid (round to 1 decimal)
            pos_key = tuple(np.round(self.robot_positions[i], 1))
            if pos_key in self.visited:
                rewards -= 30        # Revisit deterrent (prevents loitering in one spot)
            else:
                rewards += 3         # Exploration bonus (encourages sweeping the field)
            self.visited.add(pos_key)

            # ── Target coverage ──────────────────────────────────────────────────
            # Check if the robot successfully flew over an active target
            for inf_loc in list(self.infected_locations):
                if np.linalg.norm(self.robot_positions[i] - np.array(inf_loc)) <= self.infected_size:
                    self.infected_locations.remove(inf_loc)              # Mark target as cleared
                    self.infected_dict[inf_loc] = 1                      # Update dictionary for network observation
                    rewards += 1000                                      # Big reward for successfully visiting a target

            # Trajectory buffer (rendering only)
            self.trajectories[i].append(self.robot_positions[i].copy())
            if len(self.trajectories[i]) > 200:                          # Limit memory of breadcrumbs to 200 ticks
                self.trajectories[i].pop(0)

        # ── Path length computation ──────────────────────────────────
        step_dist = np.linalg.norm(self.robot_positions - self.prev_positions, axis=1) # True travel distance
        step_path = float(np.sum(step_dist))
        self.total_path_length += step_path
        self.prev_positions     = self.robot_positions.copy()

        # ── Global time and path penalties ───────────────────────────
        if self.reward_ablation != "no_path":
            rewards -= 1.0 * step_path     # Path-length penalty (encourages shortest route)
            rewards -= 2.0                 # Time penalty (encourages fast completion)

        # Distance shaping to nearest unvisited target (always active) — unchanged
        # Provides dense gradient pointing robots towards remaining targets
        if self.infected_locations:
            target_arr = np.array(self.infected_locations, dtype=np.float32)
            dists_mat  = np.linalg.norm(
                self.robot_positions[:, None] - target_arr[None, :], axis=2)
            rewards += 0.5 * float(np.sum(np.exp(-np.min(dists_mat, axis=1))))

        # ── Terminal signals ──────────────────────────────────────────────────────
        term_cond = ""
        
        # Win condition: All targets visited
        if len(self.infected_locations) == 0:
            if self.reward_ablation != "no_term":
                rewards += 5000      # Matches reference env success bonus
            term_cond  = "visited_all"
            terminated = True

        # Lose condition: Robots crashed into each other (only if not already done)
        if not terminated and self.num_robots > 1 and compute_min_dist(self.robot_positions) < self.robot_size:
            if self.reward_ablation != "no_term":
                rewards -= 5000     # Crash penalty
            term_cond  = "collision"
            terminated = True

        # Truncation: Hit max time steps
        truncated = self.step_count >= self.max_steps
        if truncated:
            term_cond = "max_steps"

        # Generate observations and metadata
        obs, info = self._get_obs()
        info.update({
            "step_count":        self.step_count,
            "remaining_targets": len(self.infected_locations),
            "path_length":       self.total_path_length,
            "term_cond":         term_cond,
        })
        return obs, rewards, terminated, truncated, info

    def render(self):
        """Displays the environment graphically using PyGame."""
        if self.render_mode != "human":
            return
            
        # Initialize PyGame window safely on first call
        if self.screen is None:
            pygame.init()
            pygame.display.init()
            self.screen = pygame.display.set_mode((self.screen_width, self.screen_height))
            pygame.display.set_caption("Multi-UAV Path Planning")
            self.clock = pygame.time.Clock()

        # Clear background to white
        self.screen.fill((255, 255, 255))
        
        # Draw the boundary polygon
        scaled_poly = [self.world_to_screen(p) for p in self.poly_vertices]
        pygame.draw.polygon(self.screen, (255, 255, 0), scaled_poly)

        # Draw trajectory tails behind each robot
        for i in range(self.num_robots):
            if len(self.trajectories[i]) > 1:
                pygame.draw.lines(self.screen, (150, 150, 150), False,
                                  [self.world_to_screen(p) for p in self.trajectories[i]], 2)

        # ── Rendering Scaling Fix ────────────────────────────────────────────────
        # Using a much larger baseline multiplier to match Code 1's visibility intent.
        # Ensure robots and targets aren't shrunken by dynamic screen scaling.
        r_px = int(self.robot_size * self.render_scale)
        
        # Draw Robots
        for i in range(self.num_robots):
            pygame.draw.circle(self.screen, self.robot_colors[i % len(self.robot_colors)],
                               self.world_to_screen(self.robot_positions[i]), r_px)

        inf_px = int(self.infected_size * self.render_scale / 1.5)
        
        # Draw unvisited targets (Cyan)
        for loc in self.infected_locations:
            pygame.draw.circle(self.screen, (0, 220, 220), self.world_to_screen(loc), inf_px)
            
        # Draw visited targets (Dark Gray)
        for loc, vis in self.infected_dict.items():
            if vis:
                pygame.draw.circle(self.screen, (80, 80, 80), self.world_to_screen(loc), inf_px)

        # Swap display buffers and tick framerate
        pygame.display.flip()
        self.clock.tick(self.metadata["render_fps"])

    def close(self):
        """Cleanly shuts down the PyGame window."""
        if self.screen is not None:
            pygame.display.quit()
            pygame.quit()
            self.screen = None


In [ ]:
# ================================
# Register and load the configuation files
# ================================
num_robots = 3
height = 0.4
if 'MultiUAV-v0' not in gym.envs.registry:
    gym.register(id='MultiUAV-v0', entry_point=MultiUAV, max_episode_steps=1000)
json_path = os.path.join('..', '..', 'exp_sets', 'uav', 'cont_sets.json')
json_dict = read_uav_json(json_path)

## Simulation

**IMPORTANT**: open the scene file using CoppeliaSim robot simulator before running the below cells

In [ ]:
# ================================
# Initialize the ZeroMQ Client
# ================================
client = RemoteAPIClient()
sim = client.getObject('sim')
defaultIdleFps = sim.getInt32Param(sim.intparam_idle_fps)
sim.setInt32Param(sim.intparam_idle_fps, 0)

# ================================
# Drone simulator class
# ================================
class Drone_simulator:
    def __init__(self, gym_env, scaling_factor=5, height=1, num_robots=3):
        self.scaling_factor = scaling_factor
        self.polygon = gym_env.unwrapped.poly_vertices
        self.scaled_polygon = [(x/scaling_factor, y/scaling_factor) for (x,y) in self.polygon]
        self.rounded_polygon = self.scaled_polygon + [self.scaled_polygon[0]]
        self.weed_locations = [tuple(loc) for loc in gym_env.unwrapped.initial_inf_locations]
        self.height = height
        self.num_robots = num_robots
        self.max_robots = 5

        # Drawing handle
        self.field_drawing = None

        # Track spawned weeds (IMPORTANT)
        self.spawned_weeds = []

        # ---- cache quadcopters & initial states ----
        self.all_drones = []
        self.initial_positions = {}
        i = 0
        while True:
            h = sim.getObject(f"/Quadcopter[{i}]", {'noError': True})
            if h == -1:
                break
            self.all_drones.append(h)
            self.initial_positions[h] = sim.getObjectPosition(h, -1)
            i += 1

    # ------------------------------------------------

    def start_simulation(self):
        self.trace_line = sim.addDrawingObject(sim.drawing_lines, 5, 0, -1, 9999, [255,0,0]) # red line
        sim.startSimulation()
        print("Simulation started")

    def stop_simulation(self):
        sim.removeDrawingObject(self.trace_line)
        sim.stopSimulation()
        while sim.getSimulationState() != sim.simulation_stopped:
            sim.step()
        
        # ---- cleanup runtime artifacts ----
        self.clear_field()
        self.clear_weeds()

        # ---- restore robots ----
        for drone, pos in self.initial_positions.items():
            sim.setObjectPosition(drone, -1, pos)
            sim.setModelProperty(drone, 0)
        print("Simulation reset completed")

    # Field drawing
    def draw_field(self):
        white = [255, 255, 255]
        self.field_drawing = sim.addDrawingObject(sim.drawing_lines, 5, 0, -1, 9999, white)

        for i in range(len(self.rounded_polygon) - 1):
            p1 = self.rounded_polygon[i]
            p2 = self.rounded_polygon[i+1]
            line = [
                p1[0], p1[1], 0.1,
                p2[0], p2[1], 0.1     # drawing at height 0.1
            ]
            sim.addDrawingObjectItem(self.field_drawing, line)
    
    def clear_field(self):
        if self.field_drawing is not None:
            sim.removeDrawingObject(self.field_drawing)
            self.field_drawing = None

    # ------------------------------------------------

    def _robot_position(self, info, robot_index):
        """Return (x, y) for robot i from gym info (array or legacy dict)."""
        pos = info[f'robot{robot_index}']
        if isinstance(pos, dict):
            pos = pos['position']
        return np.asarray(pos, dtype=float)

    def set_agent_positions(self, info):
        for i, drone in enumerate(self.all_drones):
            if i < self.num_robots:
                pos = self._robot_position(info, i)
                pos = [p / self.scaling_factor for p in pos] + [self.height]
                sim.setObjectPosition(drone, -1, pos)
                sim.setObjectInt32Param(
                    drone, sim.objintparam_visibility_layer, 1
                )
            else:
                # hide unused drones
                sim.setModelProperty(
                    drone,
                    sim.modelproperty_not_visible
                    | sim.modelproperty_not_collidable
                    | sim.modelproperty_not_detectable
                    | sim.modelproperty_not_dynamic
                )
    
    def set_weed_locations(self):
        weed_template = sim.getObject('/weed')

        # ---- clear previous weeds ----
        self.clear_weeds()

        # ---- spawn new weeds ----
        for loc in self.weed_locations:
            x = [xi / self.scaling_factor for xi in loc]
            new_pos = x + [0]

            new_weed = sim.copyPasteObjects([weed_template])[0]
            sim.setObjectPosition(new_weed, -1, new_pos)

            self.spawned_weeds.append(new_weed)
    
    def clear_weeds(self):
        for obj in self.spawned_weeds:
            if sim.isHandle(obj):
                sim.removeObject(obj)
        self.spawned_weeds = []

    # ------------------------------------------------
    # def sync_weeds_from_env(self, gym_env):
    #     """Respawn weed props for targets not yet visited in the gym env."""
    #     remaining = [tuple(loc) for loc in gym_env.unwrapped.infected_locations]
    #     if remaining != self.weed_locations:
    #         self.weed_locations = remaining
    #         self.set_weed_locations()

    def move_agents(self, info):
        """Move CoppeliaSim targets to match gym robot positions."""
        for i in range(self.num_robots):
            target = sim.getObject(f"/target[{i}]")
            prev_pos = sim.getObjectPosition(target, -1) # current object position
            pos = self._robot_position(info, i)
            pos = [p / self.scaling_factor for p in pos] + [self.height]
            sim.setObjectPosition(target, -1, pos)
            # draw the line
            line_data = prev_pos + pos
            sim.addDrawingObjectItem(self.trace_line, line_data)

# ================================
# Function to run simulation
# ================================
def run_simulation(env_id, trained_model_path):
    if not os.path.isfile(trained_model_path):
        raise FileNotFoundError(f'Trained model not found: {trained_model_path}')
    model = CrossQ.load(trained_model_path)

    # Create Gym environment
    env = gym.make('MultiUAV-v0', render_mode='human', field_info=json_dict[f"set{env_id}"], num_robots=num_robots, max_steps=1000)
    env.metadata['render_fps'] = 10
    obs, info = env.reset()
    env.render()

    # Create simulator
    # Make the simulator object, draw the field, and set agent positions
    drone_simulator = Drone_simulator(gym_env=env, scaling_factor=10, height=height, num_robots=num_robots)
    # drone_simulator.start_simulation()
    drone_simulator.draw_field()
    drone_simulator.set_agent_positions(info=info)
    drone_simulator.set_weed_locations()
    print(info)

    input("Press any key to start simulation:")

    # Start CoppeliaSim
    drone_simulator.start_simulation()
    terminated = False
    truncated = False
    total_rewards = 0
    while True:
        action, _ = model.predict(obs, deterministic=True)
        obs, reward, terminated, truncated, info = env.step(action)
        env.render()
        total_rewards += reward
        # drone_simulator.sync_weeds_from_env(env)
        print(
            f"reward={reward:.2f}, "
            f"total={total_rewards:.2f}, "
            f"remaining={info.get('remaining_targets', '?')}, "
            f"action={np.round(action, 3)}"
        )
        drone_simulator.move_agents(info)
        if terminated or truncated:
            print("Episode finished")
            break
        pygame.event.get()
        # pygame.time.wait(200)
    
    return env, drone_simulator

In [ ]:
assert False, "Please open the scene file '.\simulation\uav\coppeliasim_envs\uav_common_env.ttt' before running the following cells"

### Run simulation on the 10 environment variations

#### Environment 1

In [ ]:
# Load trained network
trained_model_path = r'..\..\trained_models\uav\May13_16_v9_uav_seed42\env1_CrossQ_uav.zip' # last model
# trained_model_path = r'..\logs\Apr20_21_v1_seed123\best_model_env1\best_model.zip'  # best model

# Start simulation!
env, drone_simulator = run_simulation(env_id=1, trained_model_path=trained_model_path)

Stop simulation (IMPORTANT! otherwise the scene file will be changed if accidentally saved.)

In [ ]:
# Stop simulation and reset environment
drone_simulator.stop_simulation()
env.close()

#### Environment 2

In [ ]:
# Load trained network
trained_model_path = r'..\..\trained_models\uav\May13_16_v9_uav_seed42\env2_CrossQ_uav.zip' # last model
# trained_model_path = r'..\logs\Apr20_21_v1_seed123\best_model_env1\best_model.zip'  # best model

# Start simulation!
env, drone_simulator = run_simulation(env_id=2, trained_model_path=trained_model_path)

Stop simulation (IMPORTANT! otherwise the scene file will be changed if accidentally saved.)

In [ ]:
# Stop simulation and reset environment
drone_simulator.stop_simulation()
env.close()

#### Environment 3

In [ ]:
# Load trained network
trained_model_path = r'..\..\trained_models\uav\May13_16_v9_uav_seed42\env3_CrossQ_uav.zip' # last model
# trained_model_path = r'..\logs\Apr20_21_v1_seed123\best_model_env1\best_model.zip'  # best model

# Start simulation!
env, drone_simulator = run_simulation(env_id=3, trained_model_path=trained_model_path)

In [ ]:
# Stop simulation and reset environment
drone_simulator.stop_simulation()
env.close()

#### Environment 4

In [ ]:
# Load trained network
trained_model_path = r'..\..\trained_models\uav\May13_16_v9_uav_seed42\env4_CrossQ_uav.zip' # last model
# trained_model_path = r'..\logs\Apr20_21_v1_seed123\best_model_env1\best_model.zip'  # best model

# Start simulation!
env, drone_simulator = run_simulation(env_id=4, trained_model_path=trained_model_path)

In [ ]:
# Stop simulation and reset environment
drone_simulator.stop_simulation()
env.close()

#### Environment 5

In [ ]:
# Load trained network
trained_model_path = r'..\..\trained_models\uav\May13_16_v9_uav_seed42\env5_CrossQ_uav.zip' # last model
# trained_model_path = r'..\logs\Apr20_21_v1_seed123\best_model_env1\best_model.zip'  # best model

# Start simulation!
env, drone_simulator = run_simulation(env_id=5, trained_model_path=trained_model_path)

In [ ]:
# Stop simulation and reset environment
drone_simulator.stop_simulation()
env.close()

#### Environment 6

In [ ]:
# Load trained network
trained_model_path = r'..\..\trained_models\uav\May13_16_v9_uav_seed42\env6_CrossQ_uav.zip' # last model
# trained_model_path = r'..\logs\Apr20_21_v1_seed123\best_model_env1\best_model.zip'  # best model

# Start simulation!
env, drone_simulator = run_simulation(env_id=6, trained_model_path=trained_model_path)

In [ ]:
# Stop simulation and reset environment
drone_simulator.stop_simulation()
env.close()

#### Environment 7

In [ ]:
# Load trained network
trained_model_path = r'..\..\trained_models\uav\May13_16_v9_uav_seed42\env7_CrossQ_uav.zip' # last model
# trained_model_path = r'..\logs\Apr20_21_v1_seed123\best_model_env1\best_model.zip'  # best model

# Start simulation!
env, drone_simulator = run_simulation(env_id=7, trained_model_path=trained_model_path)

In [ ]:
# Stop simulation and reset environment
drone_simulator.stop_simulation()
env.close()

#### Environment 8

In [ ]:
# Load trained network
trained_model_path = r'..\..\trained_models\uav\May13_16_v9_uav_seed42\env8_CrossQ_uav.zip' # last model
# trained_model_path = r'..\logs\Apr20_21_v1_seed123\best_model_env1\best_model.zip'  # best model

# Start simulation!
env, drone_simulator = run_simulation(env_id=8, trained_model_path=trained_model_path)

In [ ]:
# Stop simulation and reset environment
drone_simulator.stop_simulation()
env.close()

#### Environment 9

In [ ]:
# Load trained network
trained_model_path = r'..\..\trained_models\uav\May13_16_v9_uav_seed42\env9_CrossQ_uav.zip' # last model
# trained_model_path = r'..\logs\Apr20_21_v1_seed123\best_model_env1\best_model.zip'  # best model

# Start simulation!
env, drone_simulator = run_simulation(env_id=9, trained_model_path=trained_model_path)

In [ ]:
# Stop simulation and reset environment
drone_simulator.stop_simulation()
env.close()

#### Environment 10

In [ ]:
# Load trained network
trained_model_path = r'..\..\trained_models\uav\May13_16_v9_uav_seed42\env10_CrossQ_uav.zip' # last model
# trained_model_path = r'..\logs\Apr20_21_v1_seed123\best_model_env1\best_model.zip'  # best model

# Start simulation!
env, drone_simulator = run_simulation(env_id=10, trained_model_path=trained_model_path)

In [ ]:
# Stop simulation and reset environment
drone_simulator.stop_simulation()
env.close()

**IMPORTANT**: When closing the CoppeliaSim scene file, please select "No" for the question "Do you wish to save the changes?"